# Pure-space spectral diagnostic: smooth=0.5, nugget-free vs nugget=0

This notebook fits each hourly pure-space slice with isotropic Column Vecchia under two variants:

- `nugget_free`: estimate `sigmasq`, `range`, and `nugget`
- `nugget0`: estimate `sigmasq` and `range`, with nugget fixed at zero

Both use `smooth=0.5`, `mean_design='latlon'`, and the same every-k resolution thinning used in the previous pure-space resolution checks.

The spectral diagnostic compares the detrended empirical residual spectrum against the theoretical fitted Matern spectrum. For `nugget_free`, the theoretical spectrum includes a flat nugget floor before scaling to the empirical spectrum. Because the thinning is every-k over the ordered grid, the empirical spectrum is a masked/zero-filled periodogram and should be read as a shape diagnostic rather than an exact Whittle likelihood.


In [ ]:
import gc
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

LOCAL_SRC = '/Users/joonwonlee/Documents/GEMS_TCO-1/src'
AMAREL_SRC = '/home/jl2815/tco'
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_space_iso_050826 import (
    ColumnSpaceIsoTrendVecchiaFit,
    ColumnSpaceIsoNoNuggetTrendVecchiaFit,
)

DEVICE = torch.device('cpu')
DTYPE = torch.float64
ROUND_DECIMALS = 4

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 200)
print('SRC:', SRC)
print('device:', DEVICE)


In [ ]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX = 2                 # 0-based: 2 -> 2024-07-03
HOUR_IDX_LIST = list(range(8))
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

SMOOTH = 0.5
RESOLUTION_STRIDES = [8, 4, 2, 1]
MEAN_DESIGN = 'latlon'

COLUMN_SPEC = {
    'head_right_cols': 0,
    'above_count': 2,
    'right_col_count': 3,
    'per_time_conditioning_count': 10,
    'target_chunk_size': 512,
}

VARIANTS = {
    'nugget_free': {
        'class': ColumnSpaceIsoTrendVecchiaFit,
        'model': 'ColumnSpaceIso_Up2_Right3_M10_realloc',
        'kernel': 'column_space_iso_tonly_realloc',
        'p_labels': ['sigmasq', 'range', 'nugget'],
        'init': {'sigmasq': 13.0, 'range': 0.25, 'nugget': 2.5},
    },
    'nugget0': {
        'class': ColumnSpaceIsoNoNuggetTrendVecchiaFit,
        'model': 'ColumnSpaceIsoNoNugget_Up2_Right3_M10_realloc',
        'kernel': 'column_space_iso_nugget0_tonly_realloc',
        'p_labels': ['sigmasq', 'range'],
        'init': {'sigmasq': 13.0, 'range': 0.25},
    },
}

LBFGS_LR = 1.0
LBFGS_STEPS_FULL = 8
LBFGS_EVAL = 20
LBFGS_HIST = 10
GRAD_TOL = 1e-5

RUN_FULL = True
RUN_SPECTRUM = True

PURE_SPACE_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space')
OUT_DIR = PURE_SPACE_DIR / 'log'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = 'real_pure_space_spectral_s05_nugget_compare_column_050826'

print('day:', f'{YEAR}{MONTH:02d}{DAY_IDX + 1:02d}')
print('hours:', HOUR_IDX_LIST)
print('strides:', RESOLUTION_STRIDES)
print('smooth:', SMOOTH)
print('variants:', list(VARIANTS))


In [ ]:
def phys_to_log(init, p_labels):
    return [np.log(init[p]) for p in p_labels]


def backmap(raw, p_labels, variant):
    est = {p: float(np.exp(raw[i])) for i, p in enumerate(p_labels)}
    if 'nugget' not in est:
        est['nugget'] = 0.0
    return est


def make_full_params(variant):
    spec = VARIANTS[variant]
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(spec['init'], spec['p_labels'])]


def count_valid(input_map):
    return int(sum(((~torch.isnan(v[:, 2])) & (~torch.isnan(v[:, 0])) & (~torch.isnan(v[:, 1]))).sum().item() for v in input_map.values()))


def space_diag(model):
    groups = getattr(model, 'Batched_Groups', []) or []
    if not groups:
        return {'n_batches': 0, 'n_tails': 0, 'mean_m': 0.0, 'max_m': 0, 'largest_batch_n': 0}
    ns = np.asarray([int(g['target_idx'].shape[0]) for g in groups], dtype=np.int64)
    ms = np.asarray([int(g['max_m']) for g in groups], dtype=np.int64)
    return {
        'n_batches': int(len(groups)),
        'n_tails': int(ns.sum()),
        'mean_m': float(ms.mean()),
        'max_m': int(ms.max()),
        'largest_batch_n': int(ns.max()),
    }


def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out


def empty_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def resolution_label(stride):
    return f'x{int(stride)}'


In [ ]:
loader = load_data_dynamic_processed(config.mac_data_load_path)
df_map, _, _, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=10,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=True,
)
month_keys = sorted(df_map.keys())
day_keys = month_keys[DAY_IDX * 8:(DAY_IDX + 1) * 8]
if len(day_keys) != 8:
    raise ValueError(f'Day {DAY_IDX} has {len(day_keys)} slices, expected 8')

first_df = df_map[day_keys[0]].reset_index(drop=True)
grid_order = (
    first_df
    .assign(_orig_idx=np.arange(len(first_df)))
    .sort_values(['Longitude', 'Latitude', '_orig_idx'], kind='mergesort')['_orig_idx']
    .to_numpy(dtype=np.int64)
)
grid_coords_full = first_df.iloc[grid_order][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)

print('monthly_mean:', round(monthly_mean, 4))
print('day keys:', day_keys[0], '...', day_keys[-1])
print('full grid:', grid_coords_full.shape)
print('unique lat/lon:', len(np.unique(np.round(grid_coords_full[:,0], 6))), len(np.unique(np.round(grid_coords_full[:,1], 6))))


In [ ]:
def load_hour_map(hour_idx):
    key = day_keys[int(hour_idx)]
    hour_df_map = {key: df_map[key]}
    hour_map, _ = loader.load_working_data(
        hour_df_map,
        monthly_mean=monthly_mean,
        idx_for_datamap=[0, 1],
        ord_mm=grid_order,
        dtype=DTYPE,
        keep_ori=True,
    )
    return {k: v.to(DEVICE) for k, v in hour_map.items()}, key


def thin_hour_map(hour_map, stride):
    n_full = grid_coords_full.shape[0]
    thin_idx = np.arange(0, n_full, int(stride), dtype=np.int64)
    thin_map = {k: v[thin_idx].contiguous() for k, v in hour_map.items()}
    thin_grid = np.ascontiguousarray(grid_coords_full[thin_idx])
    return thin_map, thin_grid, thin_idx


def build_model(variant, input_map, grid_coords):
    spec = VARIANTS[variant]
    return spec['class'](
        smooth=SMOOTH,
        input_map=input_map,
        grid_coords=grid_coords,
        head_right_cols=COLUMN_SPEC['head_right_cols'],
        above_count=COLUMN_SPEC['above_count'],
        right_col_count=COLUMN_SPEC['right_col_count'],
        per_time_conditioning_count=COLUMN_SPEC['per_time_conditioning_count'],
        target_chunk_size=COLUMN_SPEC['target_chunk_size'],
        mean_design=MEAN_DESIGN,
    )


def make_base_row(variant, hour_idx, time_key, stride, n_grid, n_valid, pre_s, fit_s, loss, fit_iter, est, diag):
    spec = VARIANTS[variant]
    row = {
        'date_str': f'{YEAR}{MONTH:02d}{DAY_IDX + 1:02d}',
        'day_idx': DAY_IDX,
        'hour_idx': int(hour_idx),
        'time_key': str(time_key),
        'resolution_stride': int(stride),
        'resolution_label': resolution_label(stride),
        'variant': variant,
        'smooth': float(SMOOTH),
        'mean_design': MEAN_DESIGN,
        'fit_type': 'full',
        'model': spec['model'],
        'kernel': spec['kernel'],
        'coord_mode': 'regular-grid every-k thinning; covariance on Source_Latitude/Source_Longitude',
        'loss': float(loss),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter) + 1,
        'precompute_s': float(pre_s),
        'fit_s': float(fit_s),
        'total_s': float(pre_s + fit_s),
        'n_grid': int(n_grid),
        'n_valid': int(n_valid),
        'valid_fraction': float(n_valid / n_grid) if n_grid else np.nan,
        'est_sigmasq': float(est['sigmasq']),
        'est_range': float(est['range']),
        'est_nugget': float(est.get('nugget', 0.0)),
        **diag,
        'total_conditioning_nominal': COLUMN_SPEC['per_time_conditioning_count'],
        'head_right_cols': COLUMN_SPEC['head_right_cols'],
        'above_count': COLUMN_SPEC['above_count'],
        'right_col_count': COLUMN_SPEC['right_col_count'],
        'per_time_conditioning_count': COLUMN_SPEC['per_time_conditioning_count'],
    }
    return row


In [ ]:
def fit_full_variant(variant, hour_idx, stride):
    hour_map, time_key = load_hour_map(hour_idx)
    thin_map, thin_grid, _ = thin_hour_map(hour_map, stride)
    n_grid = int(thin_grid.shape[0])
    n_valid = count_valid(thin_map)
    print('\n' + '=' * 100)
    print(f'{variant} | smooth={SMOOTH} | hour={hour_idx + 1} | {time_key} | {resolution_label(stride)} | n_grid={n_grid:,} | n_valid={n_valid:,}')

    model = build_model(variant, thin_map, thin_grid)
    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = space_diag(model)

    params = make_full_params(variant)
    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS_FULL, grad_tol=GRAD_TOL)
    fit_s = time.time() - t_fit
    p_labels = VARIANTS[variant]['p_labels']
    est = backmap(out[:len(p_labels)], p_labels, variant)
    row = make_base_row(variant, hour_idx, time_key, stride, n_grid, n_valid, pre_s, fit_s, out[-1], fit_iter, est, diag)
    print('RESULT:', {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items() if k in ['variant','resolution_label','loss','est_sigmasq','est_range','est_nugget','total_s']})

    del model, params, opt, hour_map, thin_map
    empty_cache()
    return row


In [ ]:
fit_rows = []
if RUN_FULL:
    for variant in VARIANTS:
        for hour_idx in HOUR_IDX_LIST:
            for stride in RESOLUTION_STRIDES:
                fit_rows.append(fit_full_variant(variant, hour_idx, stride))
                tmp = round_df(pd.DataFrame(fit_rows))
                tmp.to_csv(OUT_DIR / f'{OUT_PREFIX}_full.csv', index=False, float_format=f'%.{ROUND_DECIMALS}f')

fit_df = pd.DataFrame(fit_rows)
full_path = OUT_DIR / f'{OUT_PREFIX}_full.csv'
round_df(fit_df).to_csv(full_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved full fits:', full_path)
display(round_df(fit_df))

param_rows = []
for _, row in fit_df.iterrows():
    for p in ['sigmasq', 'range', 'nugget']:
        param_rows.append({
            'date_str': row['date_str'],
            'hour_idx': int(row['hour_idx']),
            'time_key': row['time_key'],
            'resolution_stride': int(row['resolution_stride']),
            'resolution_label': row['resolution_label'],
            'variant': row['variant'],
            'smooth': row['smooth'],
            'parameter': p,
            'estimate': row[f'est_{p}'],
            'loss': row['loss'],
            'n_grid': row['n_grid'],
            'n_valid': row['n_valid'],
        })
param_df = pd.DataFrame(param_rows)
param_path = OUT_DIR / f'{OUT_PREFIX}_param_table.csv'
round_df(param_df).to_csv(param_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved param table:', param_path)


In [ ]:
# Parameter trajectories by variant.
plot_df = fit_df.copy()
plot_df['resolution_label'] = pd.Categorical(
    plot_df['resolution_label'], categories=[f'x{s}' for s in RESOLUTION_STRIDES], ordered=True
)
plot_df = plot_df.sort_values(['variant', 'hour_idx', 'resolution_label'])

fig, axes = plt.subplots(len(VARIANTS), 3, figsize=(15, 7.5), constrained_layout=True, sharex=True)
for r, (variant, sub_v) in enumerate(plot_df.groupby('variant', sort=False)):
    for c, p in enumerate(['sigmasq', 'range', 'nugget']):
        ax = axes[r, c]
        for hour_idx, sub in sub_v.groupby('hour_idx'):
            ax.plot(sub['resolution_label'].astype(str), sub[f'est_{p}'], marker='o', linewidth=1.4, label=f'h{int(hour_idx)+1}')
        ax.set_title(f'{variant}: {p}')
        ax.set_xlabel('resolution thinning')
        ax.set_ylabel('estimate')
        ax.grid(True, alpha=0.3)
axes[0, -1].legend(ncol=2, fontsize=8, title='hour')
fig.suptitle('Pure-space isotropic Column Vecchia, smooth=0.5: parameter trajectories', fontsize=14)
param_plot_path = OUT_DIR / f'{OUT_PREFIX}_parameter_trajectories.png'
fig.savefig(param_plot_path, dpi=180, bbox_inches='tight')
print('Saved parameter plot:', param_plot_path)
plt.show()

display(round_df(plot_df[['variant','hour_idx','resolution_stride','resolution_label','loss','est_sigmasq','est_range','est_nugget','n_grid','n_valid','total_s']]))


## Spectral diagnostic

The empirical spectrum uses the same thinned points as each fit. Residuals are computed after an OLS `intercept + lat + lon` detrend, placed back onto the full grid, tapered, zero-filled, FFTed, and radially averaged. For `nugget_free`, the theoretical spectrum includes the fitted flat nugget component before being scaled to the empirical curve.


In [ ]:
SPECTRAL_SMOOTH = SMOOTH
RADIAL_N_BINS = 70
RADIAL_QMAX = 0.985
APPLY_HANN = True
EPS = 1e-12

_lat_key = np.round(grid_coords_full[:, 0], 10)
_lon_key = np.round(grid_coords_full[:, 1], 10)
lat_vals = np.sort(np.unique(_lat_key))
lon_vals = np.sort(np.unique(_lon_key))
lat_to_row = {float(v): i for i, v in enumerate(lat_vals)}
lon_to_col = {float(v): i for i, v in enumerate(lon_vals)}
local_to_row = np.asarray([lat_to_row[float(v)] for v in _lat_key], dtype=np.int64)
local_to_col = np.asarray([lon_to_col[float(v)] for v in _lon_key], dtype=np.int64)
N_LAT, N_LON = len(lat_vals), len(lon_vals)
LAT_STEP = float(np.median(np.diff(lat_vals))) if N_LAT > 1 else 1.0
LON_STEP = float(np.median(np.diff(lon_vals))) if N_LON > 1 else 1.0
print('spectral full grid:', (N_LAT, N_LON), 'lat/lon step:', LAT_STEP, LON_STEP)


def detrended_residual_grid(hour_idx, stride):
    hour_map, time_key = load_hour_map(hour_idx)
    thin_map, thin_grid, thin_idx = thin_hour_map(hour_map, stride)
    arr = next(iter(thin_map.values())).detach().cpu().numpy()
    y = arr[:, 2].astype(float)
    lat = arr[:, 0].astype(float)
    lon = arr[:, 1].astype(float)
    valid = np.isfinite(y) & np.isfinite(lat) & np.isfinite(lon)
    if valid.sum() < 4:
        raise ValueError(f'Not enough valid points for hour={hour_idx}, stride={stride}')

    X = np.column_stack([
        np.ones(valid.sum()),
        lat[valid] - np.nanmean(lat[valid]),
        lon[valid] - np.nanmean(lon[valid]),
    ])
    beta, *_ = np.linalg.lstsq(X, y[valid], rcond=None)
    X_all = np.column_stack([
        np.ones(len(y)),
        lat - np.nanmean(lat[valid]),
        lon - np.nanmean(lon[valid]),
    ])
    resid = y - X_all @ beta

    grid = np.full((N_LAT, N_LON), np.nan, dtype=float)
    mask = np.zeros((N_LAT, N_LON), dtype=float)
    keep_idx = thin_idx[valid]
    rr = local_to_row[keep_idx]
    cc = local_to_col[keep_idx]
    grid[rr, cc] = resid[valid]
    mask[rr, cc] = 1.0
    return grid, mask, time_key, int(valid.sum())


def masked_periodogram(grid, mask):
    z = np.nan_to_num(grid, nan=0.0).astype(float)
    z = z - (np.sum(z * mask) / max(float(mask.sum()), 1.0))
    win = np.outer(np.hanning(z.shape[0]), np.hanning(z.shape[1])) if APPLY_HANN else np.ones_like(z)
    zw = z * win
    norm = np.sum((mask * win) ** 2)
    norm = norm if norm > EPS else 1.0
    return np.abs(np.fft.fftshift(np.fft.fft2(zw))) ** 2 / norm


def frequency_grid():
    fy = np.fft.fftshift(np.fft.fftfreq(N_LAT, d=LAT_STEP))
    fx = np.fft.fftshift(np.fft.fftfreq(N_LON, d=LON_STEP))
    omega_y = 2.0 * np.pi * fy
    omega_x = 2.0 * np.pi * fx
    OX, OY = np.meshgrid(omega_x, omega_y)
    k = np.sqrt(OX ** 2 + OY ** 2)
    return k, OX ** 2 + OY ** 2

K_RADIAL, OMEGA2 = frequency_grid()
_positive_k = K_RADIAL[np.isfinite(K_RADIAL) & (K_RADIAL > 0)]
K_MAX = float(np.quantile(_positive_k, RADIAL_QMAX))
RADIAL_BINS = np.linspace(0.0, K_MAX, RADIAL_N_BINS + 1)


def matern_spectrum_shape(sigmasq, range_, nugget, smooth):
    nu = float(smooth)
    alpha = 2.0 * nu / max(float(range_) ** 2, EPS)
    matern = float(sigmasq) * (alpha + OMEGA2) ** (-(nu + 1.0))
    return matern + max(float(nugget), 0.0)


def radial_average(surface):
    vals = np.asarray(surface, dtype=float).ravel()
    kk = K_RADIAL.ravel()
    good = np.isfinite(vals) & np.isfinite(kk) & (kk > 0) & (kk <= K_MAX)
    bin_idx = np.digitize(kk[good], RADIAL_BINS) - 1
    rows = []
    vg = vals[good]
    for b in range(RADIAL_N_BINS):
        m = bin_idx == b
        if not np.any(m):
            continue
        rows.append({
            'k_bin': b,
            'k_mid': 0.5 * (RADIAL_BINS[b] + RADIAL_BINS[b + 1]),
            'spectrum': float(np.nanmean(vg[m])),
            'n_freq': int(m.sum()),
        })
    return pd.DataFrame(rows)


In [ ]:
if RUN_SPECTRUM:
    if 'fit_df' not in globals() or fit_df.empty:
        fit_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_full.csv')

    spectral_rows = []
    for r in fit_df.itertuples(index=False):
        grid, mask, time_key, n_valid_spectrum = detrended_residual_grid(int(r.hour_idx), int(r.resolution_stride))
        data_p = masked_periodogram(grid, mask)
        theory_p = matern_spectrum_shape(r.est_sigmasq, r.est_range, r.est_nugget, SPECTRAL_SMOOTH)

        data_rad = radial_average(data_p).rename(columns={'spectrum': 'data_spectrum'})
        theory_rad = radial_average(theory_p).rename(columns={'spectrum': 'theory_spectrum'})
        merged = data_rad.merge(theory_rad[['k_bin', 'theory_spectrum']], on='k_bin', how='inner')

        d = merged['data_spectrum'].to_numpy(dtype=float)
        t = merged['theory_spectrum'].to_numpy(dtype=float)
        good = np.isfinite(d) & np.isfinite(t) & (d > 0) & (t > 0)
        scale = float(np.nanmedian(d[good] / t[good])) if good.any() else 1.0
        merged['theory_spectrum_scaled'] = merged['theory_spectrum'] * scale

        for m in merged.itertuples(index=False):
            spectral_rows.append({
                'date_str': r.date_str,
                'hour_idx': int(r.hour_idx),
                'time_key': time_key,
                'resolution_stride': int(r.resolution_stride),
                'resolution_label': r.resolution_label,
                'variant': r.variant,
                'smooth': float(r.smooth),
                'n_valid_spectrum': n_valid_spectrum,
                'est_sigmasq': float(r.est_sigmasq),
                'est_range': float(r.est_range),
                'est_nugget': float(r.est_nugget),
                'k_bin': int(m.k_bin),
                'k_mid': float(m.k_mid),
                'n_freq': int(m.n_freq),
                'data_spectrum': float(m.data_spectrum),
                'theory_spectrum_raw': float(m.theory_spectrum),
                'theory_spectrum_scaled': float(m.theory_spectrum_scaled),
                'theory_scale_to_data': scale,
            })

    spectral_df = pd.DataFrame(spectral_rows)
    spectral_path = OUT_DIR / f'{OUT_PREFIX}_radial_spectrum.csv'
    spectral_df.to_csv(spectral_path, index=False)
    print('Saved radial spectrum:', spectral_path)
    display(round_df(spectral_df.head(12)))


In [ ]:
# %%
# Resolution-aware spectrum plot:
# data spectrum is shown only below each thinning's effective Nyquist,
# while the fitted theoretical spectrum is shown on the full-grid frequency axis.
if 'spectral_df' not in globals():
    spectral_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_radial_spectrum.csv')

plot_spec = spectral_df.copy()
plot_spec['resolution_label'] = pd.Categorical(
    plot_spec['resolution_label'], categories=[f'x{s}' for s in RESOLUTION_STRIDES], ordered=True
)

# Use the largest radial bin from the full-grid calculation, then scale the visible data band by 1/stride.
FULL_K_PLOT_MAX = float(plot_spec['k_mid'].max())
plot_spec['effective_k_max'] = FULL_K_PLOT_MAX / plot_spec['resolution_stride'].astype(float)
plot_spec_low = plot_spec[plot_spec['k_mid'] <= plot_spec['effective_k_max']].copy()

avg_low_data = (
    plot_spec_low
    .groupby(['variant', 'resolution_label', 'resolution_stride', 'k_bin'], observed=False)
    .agg(
        k_mid=('k_mid', 'mean'),
        data_spectrum=('data_spectrum', 'mean'),
        n_hours=('hour_idx', 'nunique'),
        effective_k_max=('effective_k_max', 'mean'),
    )
    .reset_index()
)
low_path = OUT_DIR / f'{OUT_PREFIX}_radial_spectrum_lowfreq_only_avg.csv'
avg_low_data.to_csv(low_path, index=False)
print('Saved low-frequency data spectrum:', low_path)

avg_theory_full = (
    plot_spec
    .groupby(['variant', 'resolution_label', 'k_bin'], observed=False)
    .agg(
        k_mid=('k_mid', 'mean'),
        theory_spectrum_scaled=('theory_spectrum_scaled', 'mean'),
    )
    .reset_index()
)

variants_order = list(VARIANTS.keys())
labels_order = [f'x{s}' for s in RESOLUTION_STRIDES]
fig, axes = plt.subplots(len(variants_order), len(labels_order), figsize=(18, 8), constrained_layout=True, sharex=True, sharey='row')
for i, variant in enumerate(variants_order):
    for j, label in enumerate(labels_order):
        ax = axes[i, j]
        sub_data = avg_low_data[
            (avg_low_data['variant'] == variant)
            & (avg_low_data['resolution_label'].astype(str) == label)
            & (avg_low_data['k_mid'] > 0)
        ].copy()
        sub_theory = avg_theory_full[
            (avg_theory_full['variant'] == variant)
            & (avg_theory_full['resolution_label'].astype(str) == label)
            & (avg_theory_full['k_mid'] > 0)
        ].copy()
        if sub_data.empty or sub_theory.empty:
            ax.set_visible(False)
            continue
        k_cut = float(sub_data['effective_k_max'].iloc[0])
        ax.plot(sub_theory['k_mid'], sub_theory['theory_spectrum_scaled'], color='tab:red', linewidth=1.8, linestyle='--', label='fitted theoretical spectrum')
        hour_sub = plot_spec_low[
            (plot_spec_low['variant'] == variant)
            & (plot_spec_low['resolution_label'].astype(str) == label)
            & (plot_spec_low['k_mid'] > 0)
        ]
        for _, hs in hour_sub.groupby('hour_idx'):
            ax.plot(hs['k_mid'], hs['data_spectrum'], color='0.7', alpha=0.22, linewidth=0.7)
        ax.plot(sub_data['k_mid'], sub_data['data_spectrum'], color='black', linewidth=2.0, label='data residual spectrum')
        ax.axvline(k_cut, color='0.55', linewidth=1.0, linestyle=':', alpha=0.9)
        ax.set_xlim(0, FULL_K_PLOT_MAX)
        ax.set_title(f'{variant}, {label}  (data k <= {k_cut:.1f})')
        ax.set_xlabel('radial wavenumber on full-grid scale')
        if j == 0:
            ax.set_ylabel('spectrum')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.25)
axes[0, 0].legend(fontsize=8)
fig.suptitle(f'smooth={SMOOTH}: low-frequency data vs full theoretical spectrum', fontsize=14)
low_plot_path = OUT_DIR / f'{OUT_PREFIX}_data_lowfreq_theory_full_by_variant_resolution.png'
fig.savefig(low_plot_path, dpi=180, bbox_inches='tight')
print('Saved low-data/full-theory spectrum plot:', low_plot_path)
plt.show()
